# Descarga del Dataset

In [19]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dschettler8845/brats-2021-task1")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'brats-2021-task1' dataset.
Path to dataset files: /kaggle/input/brats-2021-task1


# Descomprimir Dataset

In [20]:
import tarfile
import os

base_path = "/kaggle/input/brats-2021-task1"
extract_path = "/kaggle/working/brats_data"

os.makedirs(extract_path, exist_ok=True)

for file in os.listdir(base_path):
    if file.endswith(".tar"):
        tar_path = os.path.join(base_path, file)
        print(f"Extracting {file}...")

        with tarfile.open(tar_path) as tar:
            tar.extractall(path=extract_path)

Extracting BraTS2021_00495.tar...
Extracting BraTS2021_Training_Data.tar...


/tmp/ipykernel_25015/2308349063.py:15: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_path)


Extracting BraTS2021_00621.tar...


# Comprobar estructura de carpetas

In [21]:
for root, dirs, files in os.walk(extract_path):
    print(root)
    print("Dirs:", dirs[:5])
    print("Files:", files[:5])
    break

/kaggle/working/brats_data
Dirs: ['BraTS2021_00491', 'BraTS2021_01265', 'BraTS2021_01174', 'BraTS2021_00684', 'BraTS2021_00550']
Files: ['BraTS2021_00495_seg.nii.gz', 'BraTS2021_00621_t2.nii.gz', 'BraTS2021_00495_t1.nii.gz', 'BraTS2021_00495_t2.nii.gz', 'BraTS2021_00621_t1ce.nii.gz']


# Creamos estructura de carpetas correctas

In [22]:
import os
import shutil

base_path = "/kaggle/working/brats_data"

# nueva carpeta limpia
clean_path = "/kaggle/working/brats_clean"
os.makedirs(clean_path, exist_ok=True)

# Mover archivos sueltos

In [23]:
for file in os.listdir(base_path):
    if file.endswith(".nii.gz"):

        # extraer ID paciente
        patient_id = file.split("_")[0] + "_" + file.split("_")[1]

        patient_dir = os.path.join(clean_path, patient_id)
        os.makedirs(patient_dir, exist_ok=True)

        src = os.path.join(base_path, file)
        dst = os.path.join(patient_dir, file)

        shutil.move(src, dst)

# Mover carpetas

In [24]:
for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)

    if os.path.isdir(folder_path):
        shutil.move(folder_path, os.path.join(clean_path, folder))

Error: Destination path '/kaggle/working/brats_clean/BraTS2021_00495/BraTS2021_00495' already exists

# Volvemos a comprobar la estructura de carpetas

In [ ]:
for root, dirs, files in os.walk(clean_path):
    print(root)
    print("Dirs:", dirs[:5])
    print("Files:", files[:5])
    break

# Vemos la carpeta de un paciente

In [ ]:
patients = os.listdir(clean_path)
patient_path = os.path.join(clean_path, patients[0])

print(patient_path)
print(os.listdir(patient_path))

# Cargamos un paciente

In [ ]:
import nibabel as nib

patient_path = "/kaggle/working/brats_clean/BraTS2021_00491"

flair = nib.load(os.path.join(patient_path, "BraTS2021_00491_flair.nii.gz")).get_fdata()
t1 = nib.load(os.path.join(patient_path, "BraTS2021_00491_t1.nii.gz")).get_fdata()
t1ce = nib.load(os.path.join(patient_path, "BraTS2021_00491_t1ce.nii.gz")).get_fdata()
t2 = nib.load(os.path.join(patient_path, "BraTS2021_00491_t2.nii.gz")).get_fdata()
seg = nib.load(os.path.join(patient_path, "BraTS2021_00491_seg.nii.gz")).get_fdata()

print("Shape:", flair.shape)

# Visualizar un slice

In [ ]:
import matplotlib.pyplot as plt

slice_idx = 80

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(flair[:, :, slice_idx], cmap="gray")
plt.title("FLAIR")

plt.subplot(1,2,2)
plt.imshow(seg[:, :, slice_idx])
plt.title("Mask")

plt.show()

# Convertir a binario

In [ ]:
import numpy as np

seg_binary = (seg > 0).astype(np.float32)

# Crear multicanal

In [ ]:
X = np.stack([flair, t1, t1ce, t2], axis=-1)
print(X.shape)  # (240, 240, 155, 4)

# Normalizar datos

In [ ]:
def normalize(x):
    return (x - np.mean(x)) / np.std(x)

X = normalize(X)